To run with higer concurrency you'll need to scale up the VM.  By default it runs with 4 cores.  To add a code cell below this that looks like:

```
%%configure -f
{
    "vCores": 16,// Recommended values: [4, 8, 16, 32, 64], Fabric will allocate matched memory according to the specified vCores.
}
```

Note that this will increase the notebook startup time as a VM will need to be spun-up.

## Version 3: several semantic models, user ramp and think times

This notebook runs **one** semantic model at **one** concurrency level. It is orchestrated by two pipelines:

- `PL_PerfTest_MultiModel_v3`: **parallel** ForEach (at most 10) over the list of models.
- `PL_PerfTest_Ramp_v3`: **sequential** ForEach over the user ramp (for example 6, 20, 60, 100, 200, 300) that calls this notebook.

Each model needs its own name (`load_test_name`), its own `dataset` and its own navigation file
(`queryfile`, for example `PowerBIPerformanceData_<model>.json`) in `Files/PerfScenarios/Queries/`.

Think times: with `enable_think_times = True` the pauses between user actions recorded in the JSON file are replayed
(`think_time_scale`, `max_think_time_sec`, `think_time_jitter_pct`). `delay_sec` remains the pause between iterations.
With think times enabled, the per-query wait of `delay_sec` is replaced by the pauses between actions.
`ramp_up_sec` spreads the start of the virtual users and `initial_user_jitter_sec` adds a random initial offset.

RLS impersonation (`EffectiveUserName`) is controlled by `rls_source` (`list`, `file` or `none`) and `rls_scope` (`global` or `per_model`):

| `rls_source` | `rls_scope = global` | `rls_scope = per_model` |
|---|---|---|
| `list` | `rls_users` (comma-separated UPNs) for every model | `users` of the model's item in the `models` parameter; if absent, `rls_users` |
| `file` | `rls_users_file` for every model | `users_file` of the model's item; if absent, `rls_users_file` |
| `none` | no impersonation | no impersonation |

A users file has one UPN per line (blank lines and `#` comments are ignored, duplicates are removed). The UPNs are spread across the
virtual users in turn. With no identities, the identity that runs the test is used.


%%configure -f
{
    "vCores": 4
}

In [ ]:
import uuid
import sempy.fabric
import pandas as pd
import polars as pl
import os
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
import sempy
import time


### Parameters
The next cell is tagged **Parameters** and holds the defaults for a manual run.
When the pipeline calls this notebook, Fabric inserts an *injected parameters* cell right below it that overrides
`load_test_name`, `dataset`, `workspace`, `queryfile`, `concurrent_threads`, `delay_sec`, `iterations`, the RLS settings and the
think-time and ramp parameters. Think times are **disabled** by default (original behaviour).

In [ ]:
load_test_name = "My test"                                                         #name of test for logging
dataset = "SEMANTIC_MODEL_NAME"                                                     #name of the semantic model to test
workspace = None                                                                    #workspace containing the semantic model to test "None" for the current workspace

queryfile = "/lakehouse/default/Files/PerfScenarios/Queries/PowerBIPerformanceData.json"    #Power BI Desktop performance analyzer file containing the use case to test
concurrent_threads = 6                                                                      #number of concurrent threads to test
delay_sec = 4                                                                               #number of seconds each user waits between iterations of the use case
iterations = 1                                                                              #number of iterations run by each user

# --- Think times (disabled by default: same behaviour as the original notebook) ---
enable_think_times = False                                                                  #replay the pauses between user actions recorded in the JSON file
think_time_scale = 1.0                                                                      #1.0 = recorded pace; 0.5 = twice as fast; 2.0 = twice as slow; 0.0 = no internal pauses
max_think_time_sec = 60.0                                                                   #cap applied to any single pause
think_time_jitter_pct = 0.0                                                                 #0.10 = +/-10 % random variation on each pause
fallback_think_time_sec = 0.0                                                               #pause between queries when the JSON file has no User Action events

# --- Staggered start (ramp inside each level) ---
ramp_up_sec = 0.0                                                                           #window over which the virtual users start; 0 = all at once
initial_user_jitter_sec = 0.0                                                               #extra random offset (0..N s) before the first action

# --- RLS identities (EffectiveUserName) ---
rls_source = "list"                                                                         #where the identities come from: "list" (rls_users / models[].users), "file" (text file, one UPN per line) or "none"
rls_scope = "global"                                                                        #"global" = the same identities for every model; "per_model" = each model uses its own (models[].users or models[].users_file)
rls_users = ""                                                                              #global list: comma-separated UPNs spread across the virtual users; empty = no impersonation
rls_users_file = ""                                                                         #global file, for example /lakehouse/default/Files/PerfScenarios/Users/users.txt (one UPN per line)
model_settings = ""                                                                         #JSON of the models[] item of this model (filled in by the pipeline); keys used here: users, users_file


In [ ]:
# Values received from the pipeline ('injected parameters' cell) or, in a manual run, those of the Parameters cell
import json
import re


def _as_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


concurrent_threads = int(concurrent_threads)
delay_sec = int(delay_sec) if float(delay_sec).is_integer() else float(delay_sec)
iterations = int(iterations)
enable_think_times = _as_bool(enable_think_times)
think_time_scale = float(think_time_scale)
max_think_time_sec = float(max_think_time_sec)
think_time_jitter_pct = float(think_time_jitter_pct)
fallback_think_time_sec = float(fallback_think_time_sec)
ramp_up_sec = float(ramp_up_sec)
initial_user_jitter_sec = float(initial_user_jitter_sec)
dataset = (dataset or "").strip()
rls_source = str(rls_source or "list").strip().lower()
rls_scope = str(rls_scope or "global").strip().lower()
rls_users_file = str(rls_users_file or "").strip()
if rls_source not in ("list", "file", "none"):
    raise ValueError(f"rls_source must be 'list', 'file' or 'none': {rls_source!r}")
if rls_scope not in ("global", "per_model"):
    raise ValueError(f"rls_scope must be 'global' or 'per_model': {rls_scope!r}")
try:
    model_settings_doc = json.loads(model_settings) if str(model_settings or "").strip() else {}
except ValueError as exc:
    raise ValueError(f"model_settings is not valid JSON: {exc}") from exc
if not isinstance(model_settings_doc, dict):
    raise ValueError("model_settings must be a JSON object (the item of the models parameter)")


def _parse_upns(value, origin):
    """UPNs from a list or from text (one per line, or separated by commas or semicolons). Comments start with '#'."""
    if isinstance(value, (list, tuple)):
        value = "\n".join(str(v) for v in value)
    found, invalid = {}, []
    for number, line in enumerate(str(value or "").splitlines(), start=1):
        line = re.sub(r"(^|\s)#.*$", "", line)
        for token in re.split(r"[,;]", line):
            token = token.strip()
            if not token:
                continue
            if re.fullmatch(r"[^@\s]+@[^@\s]+", token):
                found.setdefault(token.lower(), token)  # duplicates are removed, the first spelling is kept
            else:
                invalid.append(f"{origin}, line {number}: {token!r}")
    if invalid:
        raise ValueError("Not a valid user principal name (expected user@domain):\n  " + "\n  ".join(invalid[:10]))
    return list(found.values())


def _read_users_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Users file not found: {path}")
    with open(path, encoding="utf-8-sig") as f:
        upns = _parse_upns(f.read(), path)
    if not upns:
        raise ValueError(f"Users file contains no users: {path}")
    return upns


def resolve_rls_users():
    """Returns (UPNs, description of where they come from) according to rls_source and rls_scope."""
    if rls_source == "none":
        return [], "none (rls_source = none)"
    per_model = rls_scope == "per_model"
    if rls_source == "list":
        own = _parse_upns(model_settings_doc.get("users"), "models[].users") if per_model else []
        if own:
            return own, "list of this model (models[].users)"
        upns = _parse_upns(rls_users, "rls_users")
        return upns, "global list (rls_users)" if upns else "none (no users in the list)"
    own_file = str(model_settings_doc.get("users_file") or "").strip() if per_model else ""
    if own_file:
        return _read_users_file(own_file), f"file of this model (models[].users_file): {own_file}"
    if not rls_users_file:
        raise ValueError("rls_source = 'file' but no file was given (set rls_users_file or, with rls_scope = 'per_model', users_file in the model)")
    return _read_users_file(rls_users_file), f"global file (rls_users_file): {rls_users_file}"


rls_user_list, rls_origin = resolve_rls_users()
workspace = workspace.strip() if isinstance(workspace, str) and workspace.strip() else None

if not dataset:
    raise ValueError("dataset (semantic model) is required")
if not 1 <= concurrent_threads <= 300:
    raise ValueError(f"concurrent_threads out of range (1-300): {concurrent_threads}")
if think_time_scale < 0:
    raise ValueError("think_time_scale cannot be negative")
if max_think_time_sec < 0:
    raise ValueError("max_think_time_sec cannot be negative")
if not 0.0 <= think_time_jitter_pct <= 1.0:
    raise ValueError("think_time_jitter_pct must be between 0.0 and 1.0")
if min(ramp_up_sec, initial_user_jitter_sec, fallback_think_time_sec) < 0:
    raise ValueError("ramp_up_sec, initial_user_jitter_sec and fallback_think_time_sec cannot be negative")
if not os.path.exists(queryfile):
    raise FileNotFoundError(f"Navigation file for model '{dataset}' not found: {queryfile}")
print(f"RLS identities: {len(rls_user_list) if rls_user_list else 'none (the identity running the test is used)'}"
      f" | source: {rls_origin} | rls_source={rls_source}, rls_scope={rls_scope}")
if rls_user_list and concurrent_threads > len(rls_user_list):
    print(f"Note: {concurrent_threads} virtual users share {len(rls_user_list)} identities; they are reused in turn.")
print(f"Active parameters: model={dataset!r}, load_test_name={load_test_name!r}, concurrent_threads={concurrent_threads}, "
      f"delay_sec={delay_sec}, iterations={iterations}, think_times={enable_think_times} (scale={think_time_scale}, "
      f"max={max_think_time_sec}s, jitter={think_time_jitter_pct:.0%}), ramp_up_sec={ramp_up_sec}, queryfile={queryfile}")

workspace = workspace if workspace else notebookutils.runtime.context['currentWorkspaceName']

if 'Collapse Here':
    ts=time.strftime("%Y%m%d-%H%M%S")
    loadtestId:str = f"{load_test_name}-c{concurrent_threads}-{ts}"
    print(loadtestId)

    lakehouseId = sempy.fabric.get_lakehouse_id()
    workspaceId = sempy.fabric.get_workspace_id()

    # Path for storing CSV files during run
    folder_path = f"/lakehouse/default/Files/PerfScenarios/logs/{loadtestId}"
    # Used for writing combined logs


    #Create folder dedicated to logs from this run only
    notebookutils.fs.mkdirs(folder_path)


    users = rls_user_list  # resolved from rls_source / rls_scope; empty = no EffectiveUserName
                    # 
                    # users = ["user1@domain.com","user@domain.com","user3@domain.com"]
                

    args = { 
        "xmla_endpoint" : None, #defaults to current workspace's endpoint
        "workspace" : workspace,
        "perf_analyzer_filename" : queryfile , 
        "model" : dataset,
        "roles" : None,
        "customdata" : None,
        "effective_username" : None,
        "iterations" : iterations,
        "delay_sec" : delay_sec,
        "loadtestId" : loadtestId ,
        "threadId" : 0 ,
        "concurrent_threads" : concurrent_threads,
        "useRootDefaultLakehouse": True,
        "load_test_name" : load_test_name,
        "enable_think_times" : enable_think_times,
        "think_time_scale" : think_time_scale,
        "max_think_time_sec" : max_think_time_sec,
        "think_time_jitter_pct" : think_time_jitter_pct,
        "fallback_think_time_sec" : fallback_think_time_sec,
        "initial_user_jitter_sec" : initial_user_jitter_sec,
        "start_delay_sec" : 0.0
    }

    activity =  {
                "name": "RunPerfScenario", # activity name, must be unique
                "path": "RunPerfScenario_v3", # notebook path
                "timeoutPerCellInSeconds": 9000, # max timeout for each cell, default to 90 seconds
                "args": {"param1": "value1"}, # notebook parameters
                "workspace": None, # workspace name, default to current workspace
                "retry": 0, # max retry times, default to 0
                "retryIntervalInSeconds": 0, # retry interval, default to 0 seconds
                "dependencies": [] # list of activity names that this activity depends on
            }


    DAG = {
        "activities": [],
        "timeoutInSeconds": 43200, # max timeout for the entire pipeline, default to 12 hours
        "concurrency": 25 # max number of notebooks to run concurrently, default to 25
    }
    DAG["concurrency"] = concurrent_threads

    for i in range(concurrent_threads):
        a = activity.copy()
        a["name"] = f"{a['name']}_{i}"
        this_args = args.copy()
        if (len(users)>0):
            user = users[i%len(users)]
            this_args["effective_username"] = user 
            #this_args["customdata"] = user

        this_args["threadId"] = i
        this_args["start_delay_sec"] = ramp_up_sec * i / concurrent_threads  # staggered start
        a["args"] = this_args

        DAG["activities"].append(a)
        
    # print(DAG)
    print('running load test')
    try:

        ####################################
        ##  Run the parallel threads here ##
        ####################################

        results = notebookutils.notebook.runMultiple(DAG) 


    except Exception as e:
        raise RuntimeError(f"Load test {loadtestId} failed") from e
    print('load test complete')

    # Consolodate CSV log files of results into single dataframe
    df_list = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            polars_df = pl.read_csv(file_path)
            # Need to store Time using format that will work for Direct Lake

            polars_df = (polars_df
                .with_columns([
                    pl.col("start_time_dt").str.to_datetime(time_unit="ms").dt.truncate("1s").alias("start_time_s").dt.replace_time_zone(time_zone="UTC")
                    ])
                )

            polars_df = (polars_df
                .with_columns([
                    pl.col("start_time_dt").str.to_datetime(time_unit="ms").alias("start_time_dt").dt.replace_time_zone(time_zone="UTC")
                    ])
                )

            df_list.append(polars_df)

    combined_df = pl.concat(df_list)

In [ ]:
combined_df = combined_df.with_columns(pl.col("start_time_s").cast(pl.Datetime).alias("start_time_s_dt"))


combined_df = combined_df.with_columns(
    combined_df.group_by("start_time_s_dt")
      .agg(pl.col("duration").mean().alias("avg1"))
      .join(combined_df, on="start_time_s_dt")
      .select("avg1")
)

combined_df = combined_df.with_columns(
    combined_df.group_by("start_time_s_dt")
      .agg(pl.col("duration").quantile(0.9).alias("p90"))
      .join(combined_df, on="start_time_s_dt")
      .select("p90")
)

combined_df = combined_df.with_columns(
    combined_df.group_by("start_time_s_dt")
      .agg(pl.col("duration").quantile(0.5).alias("p50"))
      .join(combined_df, on="start_time_s_dt")
      .select("p50")
)

display(combined_df)
